# Music Structure Analysis

In [ ]:
import IPython.display as ipd
import matplotlib.pyplot as plt
import librosa
import numpy
import scipy.spatial

from mirdotcom import mirdotcom

mirdotcom.init()

In progress.

Method:

- self similarity matrix

In [ ]:
filename = mirdotcom.get_audio("brahms_hungarian_dance_5.mp3")
y, sr = librosa.load(filename)

In [ ]:
ipd.Audio(y, rate=sr)

## Chroma

In [ ]:
hop_length = 8000
n_fft = 2**14

In [ ]:
chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length, n_fft=n_fft)

In [ ]:
chroma.shape

## MFCC

In [ ]:
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=10, hop_length=hop_length, n_fft=n_fft)

In [ ]:
mfcc.shape

## Self Similarity

In [ ]:
self_similarity = scipy.spatial.distance_matrix(mfcc.T, mfcc.T, p=1)

In [ ]:
self_similarity.shape

In [ ]:
plt.figure(figsize=(8, 7))

# Bottom right plot.
ax1 = plt.axes([0.2, 0, 0.8, 0.20])
ax1.imshow(mfcc, origin="lower", aspect="auto", cmap="coolwarm")
# ax1.set_xlabel('Chroma')
ax1.set_xticks([])
ax1.set_yticks([])
# ax1.set_ylim(20)

# Top left plot.
ax2 = plt.axes([0, 0.2, 0.20, 0.8])
ax2.imshow(mfcc.T[:, ::-1], origin="lower", aspect="auto", cmap="coolwarm")
# ax2.set_ylabel('Signal 2')
ax2.set_xticks([])
ax2.set_yticks([])
# ax2.set_ylim(20)

# Top right plot.
# ax3 = plt.axes([0.2, 0.2, 0.8, 0.8], sharex=ax1, sharey=ax2)
ax3 = plt.axes([0.2, 0.2, 0.8, 0.8])
ax3.imshow(
    self_similarity.T,
    aspect="auto",
    origin="lower",
    interpolation="nearest",
    cmap="gray",
)
ax3.set_xticks([])
ax3.set_yticks([])

Path enhancement 

1. diagonal smoothing
1. multiple filtreing
1. thresholding every element
1. scaling and penalty

Path extraction

## Novelty based segmentation

checkerboard kernel along the main diagonal

Take a first-order difference along each row.

In [ ]:
similarity_delta = librosa.feature.delta(self_similarity, width=11, axis=1)

In [ ]:
similarity_delta.shape

In [ ]:
similarity_diff = abs(similarity_delta)

In [ ]:
similarity_diff = abs(numpy.diff(self_similarity))

In [ ]:
similarity_diff.shape

Sum the absolute values of the differences across columns.

In [ ]:
novelty = similarity_diff.sum(axis=0)

In [ ]:
novelty.shape

Normalize novelty curve.

In [ ]:
novelty = novelty / novelty.max()

In [ ]:
novelty.max()

In [ ]:
novelty.min()

Peak pick.

In [ ]:
min_segment_length_sec = 10
min_segment_length_frames = librosa.time_to_frames(
    10, sr=sr, hop_length=hop_length, n_fft=n_fft
)
min_segment_length_frames

In [ ]:
peaks = librosa.onset.onset_detect(
    onset_envelope=novelty,
    hop_length=hop_length,
    pre_max=11,
    post_max=11,
    pre_avg=11,
    post_avg=11,
    wait=min_segment_length_frames,
)
peaks

Plot.

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(novelty)
plt.vlines(peaks, 0, 1, colors="r")
plt.legend(("Peaks",))
plt.ylabel("Novelty function")
plt.xlabel("Time (frames)")

In [ ]:
librosa.frames_to_time(peaks, sr=sr, hop_length=hop_length)